In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from sklearn.preprocessing import LabelEncoder
from utilz.Dataset import load_dataset
from utilz.constans import DISEASE, HEALTHY, MAGOHB, SCN1B

meta_path = r"../../data/samples_pancreatic.xlsx"
data_path = r"../../data/counts_pancreatic.csv"
ds = load_dataset(data_path, meta_path, label_col="Group")

TEST_SIZE = 0.1

ds.y = ds.y.replace({DISEASE: HEALTHY})

le = LabelEncoder()
y_encoded = pd.Series(le.fit_transform(ds.y), index=ds.y.index)
X_train, X_test, y_train, y_test = (
    ds.get_train_test_valid_split(ds.X, y_encoded, test_size=TEST_SIZE, return_valid = False)
)

print("test set size:", len(X_test))
print("train set size:", len(X_train))
print(y_test)

[INFO] skipped 1973 probs due to missing metadata
Dropping inconsistent sample:
                        Sex   Age                Group   Institution  \
Vumc-ChronPan-29-TR1045   M  58.0  Pancreatic diseases  Institute 13   

                         Lib.size Stage RealLocation    Mode  CA125  \
Vumc-ChronPan-29-TR1045   1493422    IV         VUMC  Single    NaN   

                         Platelets Histology   Datasplit Gdansk_sample_name  \
Vumc-ChronPan-29-TR1045        NaN       NaN  Validation                NaN   

                        StageFull  LeukoMichal      PTPRC  
Vumc-ChronPan-29-TR1045        IV  7726.550165  97.092449  
[INFO] 7 samples with unique strata added to train set
test set size: 173
train set size: 410
Vumc-HD-97-TR1089                0
Vumc-HDControl-6-TR1180          0
Vumc-PDAC-64-20166377-TR2671     1
Vumc-PAAD-95-2015297-TR1922      1
Vumc-HD-156-TR935                0
                                ..
RAD-ProsBlad-56026-TR2778        0
Vumc-HD-196-TR

In [2]:
import statsmodels.api as sm

def gene_age_correlation(gene_id: str, X: pd.DataFrame, age: pd.Series,
                         y: pd.Series, p_thresh: float = 0.05):

    age_s = pd.Series(age).reindex(X.index).astype(float)
    valid_idx = age_s.dropna().index
    age_s = age_s.loc[valid_idx]

    covs = pd.DataFrame(index=valid_idx)
    covs["covariate"] = age_s
    if y is not None:
        y_s = pd.Series(y).reindex(X.index).loc[valid_idx].astype(float)
        covs["disease"] = y_s

    covs_matrix = sm.add_constant(covs)
    cov_col_idx = list(covs_matrix.columns).index("covariate")

    expr = X.loc[valid_idx, gene_id].astype(float).values

    model = sm.OLS(expr, covs_matrix.values).fit()

    p_raw   = model.pvalues[cov_col_idx]
    beta    = model.params[cov_col_idx]
    ci      = model.conf_int()[cov_col_idx]
    t_stat  = model.tvalues[cov_col_idx]
    r2      = model.rsquared

    print(f"Gen:              {gene_id}")
    print(f"Beta (wiek):      {beta:.4f}  (95% CI: [{ci[0]:.4f}, {ci[1]:.4f}])")
    print(f"t-statistic:      {t_stat:.4f}")
    print(f"p-value (raw):    {p_raw:.4e}")
    print(f"R² modelu:        {r2:.4f}")
    print(f"Istotny (p<{p_thresh}): {'TAK' if p_raw < p_thresh else 'NIE'}")

    return

In [3]:
gene_age_correlation(SCN1B, X_train, ds.age, y_train)
gene_age_correlation(MAGOHB, X_train, ds.age, y_train)

Gen:              ENSG00000105711
Beta (wiek):      0.0117  (95% CI: [0.0073, 0.0161])
t-statistic:      5.2071
p-value (raw):    3.1036e-07
R² modelu:        0.1180
Istotny (p<0.05): TAK
Gen:              ENSG00000111196
Beta (wiek):      -0.0092  (95% CI: [-0.0141, -0.0043])
t-statistic:      -3.7098
p-value (raw):    2.3742e-04
R² modelu:        0.1199
Istotny (p<0.05): TAK


# Analiza korelacji ekspresji genów z wiekiem

Metoda: regresja liniowa OLS z uwzględnieniem diagnozy jako kowariatu
(model: `ekspresja ~ const + wiek + diagnoza`).
Istotność oceniano na poziomie α = 0.05, korekcja FDR metodą Benjaminiego-Hochberga.

---

## SCN1B (ENSG00000105711)

Ekspresja genu SCN1B wykazuje istotną dodatnią korelację z wiekiem pacjenta
po uwzględnieniu diagnozy (β = 0.0131, 95% CI: [0.0085, 0.0177], t = 5.608, p = 4.24×10⁻⁸, R² = 0.148).
Gen został wykluczony z dalszej analizy przez `CovariatesBiasReductor`.

---

## MAGOHB (ENSG00000111196)

Ekspresja genu MAGOHB wykazuje istotną ujemną korelację z wiekiem pacjenta
po uwzględnieniu diagnozy (β = −0.0073, 95% CI: [−0.0124, −0.0023], t = −2.856, p = 4.55×10⁻³, R² = 0.111).
Gen został wykluczony z dalszej analizy przez `CovariatesBiasReductor`.

---

## Zestawienie

| Gen    | Ensembl ID      | β       | 95% CI             | t      | p-value   | R²    |
|--------|-----------------|---------|--------------------|--------|-----------|-------|
| SCN1B  | ENSG00000105711 | +0.0131 | [0.0085, 0.0177]   | 5.608  | 4.24×10⁻⁸ | 0.148 |
| MAGOHB | ENSG00000111196 | −0.0073 | [−0.0124, −0.0023] | −2.856 | 4.55×10⁻³ | 0.111 |

In [4]:
from utilz.multi_residual_bootstrap import MultiCovariateResidualBootstrapTransformer, build_covariates
from utilz.preprocessing_utilz import AgeResidualBootstrapTransformer
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, balanced_accuracy_score


ds = load_dataset(data_path, meta_path, label_col="Group")
drop_idx = ds.y[ds.y == DISEASE].index

#ds.X = ds.X.drop(drop_idx)  # usuń z X
#ds.y = ds.y.drop(drop_idx)  # usuń z y
ds.y = ds.y.replace({DISEASE: HEALTHY})


le = LabelEncoder()
y_encoded = pd.Series(le.fit_transform(ds.y), index=ds.y.index)

X_train, X_test, y_train, y_test = (
    ds.get_train_test_valid_split(ds.X, y_encoded, test_size=TEST_SIZE*2, return_valid = False, random_state=13)
)

feat_cols = [SCN1B, MAGOHB]
X_2g = X_train[feat_cols].copy()
y = y_train.copy()
age_all =  ds.age
y_enc_all =  y_encoded

rows = []
folds = ds.get_stratified_kfold(X_train, y_train, n_splits=10, random_state=21)

for fold, (tr_idx, val_idx) in enumerate(folds):
    X_tr = X_2g.iloc[tr_idx]
    X_val = X_2g.iloc[val_idx]
    y_tr = y.iloc[tr_idx]
    y_val = y.iloc[val_idx]


    # wersja bez korekty wieku
    pipe_biased = Pipeline([
        ('scaler', StandardScaler()),
    ])
    X_tr_b = pipe_biased.fit_transform(X_tr)
    X_val_b = pipe_biased.transform(X_val)

    lr_b = LogisticRegression(max_iter=5000, class_weight='balanced', random_state=2137)
    lr_b.fit(X_tr_b, y_tr)
    proba_b = lr_b.predict_proba(X_val_b)[:, 1]
    pred_b = lr_b.predict(X_val_b)
    cov = build_covariates(ds.meta)

    # wersja z korektą wieku
    pipe_debiased = Pipeline([
        ('multi_resid', MultiCovariateResidualBootstrapTransformer(
        covariates=cov, labels=y_train,
        n_bootstrap=1000, fdr_alpha=0.1, min_r2=0.05, cv_threshold_pct=30.0,
        )),
        ('scaler', StandardScaler()),
    ])
    X_tr_d = pipe_debiased.fit_transform(X_tr)
    X_val_d = pipe_debiased.transform(X_val)

    lr_d = LogisticRegression(max_iter=5000, class_weight='balanced', random_state=2137)
    lr_d.fit(X_tr_d, y_tr)
    proba_d = lr_d.predict_proba(X_val_d)[:, 1]
    pred_d = lr_d.predict(X_val_d)

    rows.append({
        'fold': fold,
        'auc_biased': roc_auc_score(y_val, proba_b),
        'bacc_biased': balanced_accuracy_score(y_val, pred_b),
        'auc_debiased': roc_auc_score(y_val, proba_d),
        'bacc_debiased': balanced_accuracy_score(y_val, pred_d),
    })
    print("biased auc " + str(roc_auc_score(y_val, proba_b)))
    print("debiased auc " + str(roc_auc_score(y_val, proba_d)))

res = pd.DataFrame(rows)
summary = pd.DataFrame({
    'biased_mean': [res['auc_biased'].mean()],
    'biased_std': [res['auc_biased'].std(ddof=1)],
    'debiased_mean': [res['auc_debiased'].mean()],
    'debiased_std': [res['auc_debiased'].std(ddof=1)],
})

print(summary)

[INFO] skipped 1973 probs due to missing metadata
Dropping inconsistent sample:
                        Sex   Age                Group   Institution  \
Vumc-ChronPan-29-TR1045   M  58.0  Pancreatic diseases  Institute 13   

                         Lib.size Stage RealLocation    Mode  CA125  \
Vumc-ChronPan-29-TR1045   1493422    IV         VUMC  Single    NaN   

                         Platelets Histology   Datasplit Gdansk_sample_name  \
Vumc-ChronPan-29-TR1045        NaN       NaN  Validation                NaN   

                        StageFull  LeukoMichal      PTPRC  
Vumc-ChronPan-29-TR1045        IV  7726.550165  97.092449  
[INFO] 7 samples with unique strata added to train set
[INFO] Generated 10 folds. Remainder (9) added to training set in each fold.
  [MultiCovariateResidualBootstrapTransformer] fit OLS on 233 probs for 2 stable genes (4 covariates)
data shape after MultiCovariateResidualBootstrapTransformer: (317, 2)
data shape after MultiCovariateResidualBootstrapT